# Preprocesamiento de Datos — analisiscrudo.ipynb

**Entrada**: `data-1747321685273.csv` (log de interacciones de plataforma educativa)  
**Salida**: `dfNivelEjercicio`, `dfUsuario`, `dfSegmento`, transiciones (`df_1_to_2`, `df_2_to_3`)

## Cambios respecto a versión anterior
- **FIX**: Orden de procesamiento cuando `tryStep` y `completeContent` comparten timestamp.  
  Se aplica sort secundario por `_verb_order` para garantizar que `tryStep` se procese antes de `completeContent`.
- **FIX**: `asignar_sesiones` vectorizado para evitar pérdida de columna `userid` en pandas >= 2.x.
- **FIX**: Columna `_challengeId_raw` convertida a `object` antes de asignar strings (`RESET`).
- **Renombrado**: `dfCongreso` → `dfUsuario`, `dfSecuencia` → `dfSegmento`.

# 1. Carga y Parseo del CSV

**Entrada**: CSV crudo con todas las interacciones del sistema.  
**Proceso**: Se filtra año 2025, se parsea la columna JSON `extra` y se expande en columnas planas con prefijo `ext_`.

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

file_path = "data-1747321685273.csv"  # <-- ajustar ruta según necesidad
perfil = pd.read_csv(file_path, low_memory=False)

perfil['timestamp'] = pd.to_datetime(perfil['timestamp'], format='mixed')
perfil_2025 = perfil[perfil['timestamp'].dt.year == 2025].copy()

print(f"Filas totales: {len(perfil)}, Filas 2025: {len(perfil_2025)}")

In [ ]:
def parse_json(extras_str):
    try:
        return json.loads(extras_str)
    except (json.JSONDecodeError, TypeError):
        return {}

def flatten_dict(d: dict, parent_key: str = '') -> dict:
    items = {}
    for key, value in d.items():
        base = f"ext_{parent_key}" if parent_key else "ext"
        new_key = f"{base}_{key}"
        if isinstance(value, dict):
            items.update(flatten_dict(value, new_key))
        elif isinstance(value, list):
            for elem in value:
                if isinstance(elem, dict):
                    items.update(flatten_dict(elem, new_key))
                else:
                    items[new_key] = elem
        else:
            items[new_key] = value
    return items

perfil_2025['extras_dict'] = perfil_2025['extra'].apply(parse_json)
flat_series = perfil_2025['extras_dict'].apply(flatten_dict)
extras_expandidos = pd.DataFrame(flat_series.tolist(), index=perfil_2025.index)

perfil_expanded = pd.concat([perfil_2025, extras_expandidos], axis=1)
perfil_expanded.drop(columns=['extras_dict'], inplace=True)

resp_cols = [c for c in perfil_expanded.columns if c.startswith('ext_responses_')]
print(f"CSV cargado: {len(perfil_expanded)} filas (2025). Columnas 'responses' aplanadas: {resp_cols}")

# 2. Creación de dfNivelEjercicio

**Entrada**: `perfil_expanded`  
**Proceso**:
- Asignar grupo de intervención (OLM / OSLM / OSLM+VP) desde columna `tags`
- Calcular dirección de comparación social
- Propagar pollValue, progressMe, progressGroup, pollCount, inChallenge
- Asignar sesiones
- Extraer bloques de ejercicios (loadContent → completeContent)

## 2.1 Grupo de intervención y dirección de comparación

In [ ]:
perfil = perfil_expanded.copy()
perfil["timestamp"] = pd.to_datetime(perfil["timestamp"])
perfil = perfil.sort_values(["userid", "timestamp"])

# --- Grupo de intervención ---
def parse_tags(tag_str):
    if pd.isna(tag_str):
        return set()
    items = tag_str.strip("{}").split(",")
    items = [item.strip().lower() for item in items if item.strip() != ""]
    items = ["oslm" if item == "olsm" else item for item in items]
    return set(items)

def categorize(tags_set):
    has_oslm = "oslm" in tags_set
    has_motiv = "motiv-msg" in tags_set
    if has_oslm and has_motiv:
        return "OSLM+VP"
    elif has_oslm:
        return "OSLM"
    else:
        return "OLM"

grupo_por_estudiante = (
    perfil.groupby("userid", sort=False)["tags"]
    .apply(lambda tags: categorize(set().union(*tags.apply(parse_tags))))
    .reset_index(name="grupo_intervencion")
)
perfil = perfil.merge(grupo_por_estudiante, on="userid", how="left")

# --- Dirección de comparación ---
def calcular_direccion(row):
    me = row.get("ext_progressMe", np.nan)
    grp = row.get("ext_progressGroup", np.nan)
    if pd.isna(me) or pd.isna(grp) or grp == -1:
        return np.nan
    if me < grp:
        return "hacia arriba"
    if me > grp:
        return "hacia abajo"
    return "igual"

perfil["direccionComparacion"] = perfil.apply(calcular_direccion, axis=1)

# Propagar grupo_intervencion y direccionComparacion
perfil["grupo_intervencion"] = (
    perfil.groupby("userid", sort=False)["grupo_intervencion"].ffill().bfill()
)
perfil["direccionComparacion"] = (
    perfil.groupby("userid", sort=False)["direccionComparacion"].ffill().bfill()
)

print(f"Grupos: {perfil.groupby('userid')['grupo_intervencion'].first().value_counts().to_dict()}")

## 2.2 Propagación de pollValue, progressMe, progressGroup y pollCount

In [ ]:
perfil["pollValue_raw"] = perfil["ext_ext_responses_response"].where(
    perfil.verbName == "pollResponse"
)
perfil["progressMe_raw"] = perfil["ext_progressMe"].where(
    perfil.verbName == "selectSubtopic"
)
perfil["progressGroup_raw"] = perfil["ext_progressGroup"].where(
    perfil.verbName == "selectSubtopic"
)

perfil[["pollValue", "progressMe", "progressGroup"]] = (
    perfil.groupby(["userid", "topiclabel"], sort=False)[
        ["pollValue_raw", "progressMe_raw", "progressGroup_raw"]
    ]
    .ffill()
    .rename(columns={
        "pollValue_raw": "pollValue",
        "progressMe_raw": "progressMe",
        "progressGroup_raw": "progressGroup",
    })
)
perfil.drop(columns=["pollValue_raw", "progressMe_raw", "progressGroup_raw"], inplace=True)

# --- Propagar pollCount ---
perfil["pollCount"] = (perfil.verbName == "pollResponse").astype(int)
perfil["pollCount"] = (
    perfil.groupby(["userid", "topiclabel"], sort=False)["pollCount"]
    .cumsum()
    .ffill()
    .fillna(0)
    .astype(int)
)

print("pollValue, progressMe, progressGroup y pollCount propagados.")

## 2.3 Propagación de inChallenge

`challengeLoad` indica que la actividad ocurre en un desafío. Se propaga el `challengeId` hacia adelante hasta que aparezca un `selectTopic` (el usuario volvió al menú).

In [ ]:
# FIX: convertir a object dtype antes de asignar string "RESET"
perfil["_challengeId_raw"] = perfil["challengeId"].astype(object)
perfil.loc[perfil["verbName"] != "challengeLoad", "_challengeId_raw"] = np.nan

mask_select_topic = perfil["verbName"] == "selectTopic"
perfil.loc[mask_select_topic, "_challengeId_raw"] = "RESET"

perfil["inChallenge"] = (
    perfil.groupby("userid", sort=False)["_challengeId_raw"].ffill()
)
perfil.loc[perfil["inChallenge"] == "RESET", "inChallenge"] = np.nan
perfil.drop(columns=["_challengeId_raw"], inplace=True)

print(f"inChallenge no-null: {perfil['inChallenge'].notna().sum()}")

## 2.4 Funciones auxiliares y asignación de sesiones

Una sesión nueva comienza con `OpenTemplateApplication` o un gap >= 3 horas entre eventos.

In [ ]:
# --- Helpers para attempts/hints ---
def _safe_number(x, default=0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default

def _from_extra(row, key):
    try:
        raw = row.get("extra", None)
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            return None
        data = json.loads(raw) if isinstance(raw, str) else raw
        if isinstance(data, dict):
            return data.get(key, None)
        return None
    except Exception:
        return None

def get_attempts(row):
    v = row.get("ext_attempts", np.nan)
    if pd.isna(v):
        v = row.get("ext_attemps", np.nan)  # typo en datos originales
    if pd.isna(v):
        v = _from_extra(row, "attempts")
    return int(_safe_number(v, 0))

def get_hints(row):
    v = row.get("ext_hints", np.nan)
    if pd.isna(v):
        v = _from_extra(row, "hints")
    return int(_safe_number(v, 0))

def contar_ramas_ejercicio(row):
    """Número de pasos únicos del ejercicio desde extra['steps'] de completeContent."""
    raw = row.get("extra", None)
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return np.nan
    try:
        data = json.loads(raw) if isinstance(raw, str) else raw
    except Exception:
        return np.nan
    if isinstance(data, dict) and "steps" in data and isinstance(data["steps"], dict):
        return len(data["steps"])
    return np.nan

# --- Asignación de sesiones (vectorizado) ---
# FIX: Se evita groupby().apply() que en pandas >=2.x elimina la columna de agrupación.
perfil = perfil.sort_values(["userid", "timestamp"]).reset_index(drop=True)
perfil["_gap_hours"] = (
    perfil.groupby("userid")["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(3600)
    .fillna(0)
)
perfil["_is_new_session"] = (
    (perfil["verbName"] == "OpenTemplateApplication") |
    (perfil["_gap_hours"] >= 3)
).astype(int)
perfil["session"] = perfil.groupby("userid")["_is_new_session"].cumsum()
perfil.drop(columns=["_gap_hours", "_is_new_session"], inplace=True)

print(f"Sesiones asignadas. Total sesiones únicas: {perfil['session'].nunique()}")

In [ ]:
perfil

## 2.5 Extracción de bloques de ejercicios

Un bloque = secuencia `loadContent` → `tryStep`/`requestHint` → `completeContent`.  
Cada bloque produce una fila en `dfNivelEjercicio` con las métricas del ejercicio.

**FIX aplicado**: Sort secundario por `_verb_order` para que `tryStep` siempre se procese antes de `completeContent` cuando comparten timestamp.

In [ ]:
def extract_blocks_usuario(df):
    # === FIX: forzar completeContent DESPUÉS de tryStep en mismo timestamp ===
    verb_order = {'loadContent': -1, 'tryStep': 0, 'requestHint': 0, 'completeContent': 1}
    df = df.copy()
    df['_verb_order'] = df['verbName'].map(verb_order).fillna(0.5)
    df = df.sort_values(['timestamp', '_verb_order']).drop(columns='_verb_order')

    bloques = []
    topic_nav = subtopic_nav = 0
    last_shownMsg = last_progressMe = last_progressGroup = None
    last_selectionRating = None
    last_ext_ext_selectionData_optionTitle = None
    poll_history_topic = {}

    for _, row in df.iterrows():
        v = row.verbName
        uid = row.userid
        topic = row.topiclabel
        key = (uid, topic)

        if key not in poll_history_topic:
            poll_history_topic[key] = []

        if v == "pollResponse" and pd.notna(row.get("ext_ext_responses_response", np.nan)):
            poll_history_topic[key].append(row.ext_ext_responses_response)

        if v == "selectTopic":
            topic_nav += 1
        elif v == "selectSubtopic":
            subtopic_nav += 1
            last_shownMsg = row.get("ext_shownMsg", None)
            last_progressMe = row.get("ext_progressMe", None)
            last_progressGroup = row.get("ext_progressGroup", None)
        elif v == "selectionRating":
            last_selectionRating = row.result
        elif v == "displaySelection":
            last_ext_ext_selectionData_optionTitle = row.get(
                "ext_ext_selectionData_optionTitle", None
            )

        if v == "loadContent":
            bloques.append({
                "id": row.get("id", np.nan),
                "userid": uid,
                "topiclabel": topic,
                "contentCode": row.get("contentcode", np.nan),
                "contentId": row.get("contentId", np.nan),
                "start_ts": row.timestamp,
                "end_ts": None,
                "total_hints": 0,
                "pollValue": row.get("pollValue", np.nan),
                "pollCount": row.get("pollCount", np.nan),
                "session": row.session,
                "selectTopic_count": topic_nav,
                "selectSubtopic_count": subtopic_nav,
                "last_shownMsg": last_shownMsg,
                "last_progressMe": last_progressMe,
                "last_progressGroup": last_progressGroup,
                "selectionRating_result": last_selectionRating,
                "displaySelection_optionTitle": last_ext_ext_selectionData_optionTitle,
                "completed": False,
                "trayectoria": [],
                "poll_history_topic": poll_history_topic[key].copy(),
                "total_attempts": np.nan,
                "first_attempts_correct": np.nan,
                "unique_steps": np.nan,
                "grupo_intervencion": row.grupo_intervencion,
                "direccionComparacion": row.direccionComparacion,
                "inChallenge": row.get("inChallenge", np.nan),
            })

        elif bloques:
            estado = bloques[-1]

            if v == "tryStep":
                estado["trayectoria"].append({
                    "evento": "tryStep",
                    "timestamp": row.timestamp,
                    "stepID": row.get("stepID", np.nan),
                    "attempts": get_attempts(row),
                    "hints": get_hints(row),
                })
            elif v == "requestHint":
                estado["trayectoria"].append(
                    {"evento": "requestHint", "timestamp": row.timestamp}
                )

            if v == "completeContent":
                # FIX: Obtener stepIDs válidos del ejercicio desde extra["steps"]
                # para excluir trySteps fantasma (step siguiente disparado al mismo ms)
                valid_step_ids = None
                total_steps_real = contar_ramas_ejercicio(row)
                estado["unique_steps"] = total_steps_real
                try:
                    raw_extra = row.get("extra", None)
                    if raw_extra and isinstance(raw_extra, str):
                        _data = json.loads(raw_extra)
                        if isinstance(_data, dict) and "steps" in _data and isinstance(_data["steps"], dict):
                            valid_step_ids = set(_data["steps"].keys())
                except Exception:
                    pass

                all_try_steps = [t for t in estado["trayectoria"] if t["evento"] == "tryStep"]
                request_hints = [t for t in estado["trayectoria"] if t["evento"] == "requestHint"]

                # Filtrar trySteps: solo los que pertenecen al ejercicio actual
                if valid_step_ids is not None:
                    try_steps = [t for t in all_try_steps
                                 if not pd.isna(t.get("stepID", np.nan))
                                 and str(int(t["stepID"])) in valid_step_ids]
                else:
                    try_steps = all_try_steps

                estado["total_attempts"] = len(try_steps)
                estado["total_hints"] = len(request_hints)

                by_step = {}
                for t in try_steps:
                    sid = t.get("stepID", np.nan)
                    if pd.isna(sid):
                        continue
                    s = by_step.setdefault(sid, {"max_attempts": 0, "max_hints": 0})
                    s["max_attempts"] = max(s["max_attempts"], int(t.get("attempts", 0)))
                    s["max_hints"] = max(s["max_hints"], int(t.get("hints", 0)))

                estado["first_attempts_correct"] = int(
                    sum(1 for s in by_step.values()
                        if s["max_attempts"] == 0 and s["max_hints"] == 0)
                )

                if not pd.isna(total_steps_real):
                    estado["first_attempts_correct"] = min(
                        estado["first_attempts_correct"], int(total_steps_real)
                    )

                estado["end_ts"] = row.timestamp
                estado["completed"] = True
                estado["selectionRating_result"] = last_selectionRating

                topic_nav = subtopic_nav = 0
                last_shownMsg = last_progressMe = last_progressGroup = last_selectionRating = None
                last_ext_ext_selectionData_optionTitle = None

            if v in ("loadContent", "OpenTemplateApplication") and not estado["completed"]:
                estado["end_ts"] = None
                estado["completed"] = False
                topic_nav = subtopic_nav = 0
                last_shownMsg = last_progressMe = last_progressGroup = last_selectionRating = None
                last_ext_ext_selectionData_optionTitle = None

    if bloques and bloques[-1]["end_ts"] is None:
        bloques[-1]["completed"] = False

    return pd.DataFrame(bloques)

# FIX: Iterar manualmente en lugar de groupby().apply() para preservar 'userid'
all_bloques = []
for uid, gdf in perfil.groupby("userid", sort=False):
    blk_df = extract_blocks_usuario(gdf)
    if len(blk_df) > 0:
        all_bloques.append(blk_df)
result = pd.concat(all_bloques, ignore_index=True) if all_bloques else pd.DataFrame()

columnas_finales = [
    "id", "userid", "session", "topiclabel",
    "contentCode", "contentId",
    "start_ts", "end_ts",
    "total_hints",
    "pollValue", "pollCount",
    "selectTopic_count", "selectSubtopic_count",
    "last_shownMsg", "last_progressMe", "last_progressGroup",
    "selectionRating_result", "displaySelection_optionTitle",
    "completed", "trayectoria", "poll_history_topic",
    "total_attempts", "first_attempts_correct", "unique_steps",
    "grupo_intervencion", "direccionComparacion", "inChallenge",
]

dfNivelEjercicio = result[columnas_finales]
print(f"dfNivelEjercicio creado: {len(dfNivelEjercicio)} filas (incluye incompletos)")

# 3. Propagación de Valores en dfNivelEjercicio

**Proceso**:
- Propagar `last_shownMsg`, `last_progressMe`, `last_progressGroup` con ffill por (userid, session, topiclabel)
- Filtrar solo ejercicios completados
- Calcular `prop_first_attempts_correct`

In [ ]:
dfNivelEjercicio = dfNivelEjercicio.sort_values(
    by=['userid', 'session', 'topiclabel', 'start_ts']
)

dfNivelEjercicio[['last_shownMsg', 'last_progressMe', 'last_progressGroup']] = (
    dfNivelEjercicio
    .groupby(['userid', 'session', 'topiclabel'])[
        ['last_shownMsg', 'last_progressMe', 'last_progressGroup']
    ]
    .ffill()
)

# Filtrar solo ejercicios completados
dfNivelEjercicio = dfNivelEjercicio[dfNivelEjercicio['completed'] == True].copy()

# prop_first_attempts_correct = first_attempts_correct / unique_steps
dfNivelEjercicio['prop_first_attempts_correct'] = np.where(
    dfNivelEjercicio['unique_steps'] > 0,
    dfNivelEjercicio['first_attempts_correct'] / dfNivelEjercicio['unique_steps'],
    np.nan
)
dfNivelEjercicio['prop_first_attempts_correct'] = (
    dfNivelEjercicio['prop_first_attempts_correct']
    .replace([np.inf, -np.inf], np.nan)
)

dfNivelEjercicio["pollValue_num"] = pd.to_numeric(
    dfNivelEjercicio["pollValue"], errors="coerce"
)

print(f"dfNivelEjercicio tras filtrar completados: {len(dfNivelEjercicio)} filas")
dfNivelEjercicio.head()

# 4. Construcción de dfUsuario

**Entrada**: `dfNivelEjercicio` (solo completados)  
**Proceso**: Agregar métricas por usuario — engagement, desafío, ayuda, satisfacción, comparaciones upward/downward, performance y tiempos.

In [ ]:
dfUsuario = dfNivelEjercicio[["userid", "grupo_intervencion"]].drop_duplicates(
    subset="userid"
).copy()

# --- Agregaciones por usuario ---
suma_intentos = dfNivelEjercicio.groupby("userid")["total_attempts"].sum(min_count=1)
dfUsuario["total_attempts"] = dfUsuario["userid"].map(suma_intentos)

suma_primera = dfNivelEjercicio.groupby("userid")["first_attempts_correct"].sum(min_count=1)
dfUsuario["first_attempts_correct"] = dfUsuario["userid"].map(suma_primera)

pasos_reales = dfNivelEjercicio.groupby("userid")["unique_steps"].sum(min_count=1)
dfUsuario["unique_steps"] = dfUsuario["userid"].map(pasos_reales)

numero_sesiones = dfNivelEjercicio.groupby("userid")["session"].nunique()
dfUsuario["numero_sesiones_diferentes"] = dfUsuario["userid"].map(numero_sesiones)

dfUsuario["Tasa_exito"] = (
    pd.to_numeric(dfUsuario["first_attempts_correct"], errors="coerce") /
    pd.to_numeric(dfUsuario["total_attempts"], errors="coerce")
)

autoeficacia = (
    dfNivelEjercicio[dfNivelEjercicio["pollValue_num"].notna()]
    .groupby("userid")["pollValue_num"].mean()
)
dfUsuario["autoeficacia_general"] = dfUsuario["userid"].map(autoeficacia)

progreso_me = dfNivelEjercicio.groupby("userid")["last_progressMe"].mean()
dfUsuario["progreso_inicial"] = dfUsuario["userid"].map(progreso_me)

progreso_grp = dfNivelEjercicio.groupby("userid")["last_progressGroup"].mean()
dfUsuario["progreso_inicial_grupo"] = dfUsuario["userid"].map(progreso_grp)

numero_hints = dfNivelEjercicio.groupby("userid")["total_hints"].sum(min_count=1)
dfUsuario["numero_hints"] = dfUsuario["userid"].map(numero_hints)

dfUsuario["tasa_hints"] = (
    pd.to_numeric(dfUsuario["numero_hints"], errors="coerce") /
    pd.to_numeric(dfUsuario["total_attempts"], errors="coerce")
)

rating = dfNivelEjercicio.groupby("userid")["selectionRating_result"].mean()
dfUsuario["selectionRating_result"] = dfUsuario["userid"].map(rating)

# Tiempo total
# Asegurar que timestamps sean datetime
dfNivelEjercicio["start_ts"] = pd.to_datetime(dfNivelEjercicio["start_ts"])
dfNivelEjercicio["end_ts"] = pd.to_datetime(dfNivelEjercicio["end_ts"])
dfNivelEjercicio["duracion_segundos"] = (
    dfNivelEjercicio["end_ts"] - dfNivelEjercicio["start_ts"]
)
tiempo_total = dfNivelEjercicio.groupby("userid")["duracion_segundos"].sum()
dfUsuario["tiempo_total_segundos"] = dfUsuario["userid"].map(tiempo_total)

# Primer autoeficacia
df_ordenado = dfNivelEjercicio.sort_values(["userid", "session", "topiclabel", "start_ts"])
df_primer_poll = df_ordenado.groupby(
    ["userid", "session", "topiclabel"], as_index=False
).first()
df_primer_poll = df_primer_poll[["userid", "pollValue_num"]].rename(
    columns={"pollValue_num": "primer_pollValue"}
)
primer_pollValue = df_primer_poll.groupby("userid")["primer_pollValue"].mean()
dfUsuario["primer_pollValue"] = dfUsuario["userid"].map(primer_pollValue)

# --- Proporciones seguras ---
dfUsuario["unique_steps"] = pd.to_numeric(dfUsuario["unique_steps"], errors="coerce")
dfUsuario["first_attempts_correct"] = pd.to_numeric(
    dfUsuario["first_attempts_correct"], errors="coerce"
)
dfUsuario["first_attempts_correct"] = np.minimum(
    dfUsuario["first_attempts_correct"], dfUsuario["unique_steps"]
)

den = dfUsuario["unique_steps"]
num = dfUsuario["first_attempts_correct"]

dfUsuario["prop_first_attempts_correct"] = 0.0
mask_den = den > 0
dfUsuario.loc[mask_den, "prop_first_attempts_correct"] = num[mask_den] / den[mask_den]

# Casos
casos = dfNivelEjercicio.groupby("userid").size()
dfUsuario["casos"] = dfUsuario["userid"].map(casos)

print(f"dfUsuario base creado: {len(dfUsuario)} usuarios")

## 4.1 Comparaciones upward / downward y performance

In [ ]:
umbral = 0.01

mask_arriba = (
    (dfNivelEjercicio["last_progressGroup"] != -1) &
    ((dfNivelEjercicio["last_progressMe"] - dfNivelEjercicio["last_progressGroup"]) > umbral)
)
mask_abajo = (
    (dfNivelEjercicio["last_progressGroup"] != -1) &
    ((dfNivelEjercicio["last_progressMe"] - dfNivelEjercicio["last_progressGroup"]) < -umbral)
)

# Pasos y casos
dfUsuario["pasos_comparacion_arriba"] = (
    dfNivelEjercicio[mask_arriba].groupby("userid")["total_attempts"].sum()
    .reindex(dfUsuario["userid"]).fillna(0).values
)
dfUsuario["pasos_comparacion_abajo"] = (
    dfNivelEjercicio[mask_abajo].groupby("userid")["total_attempts"].sum()
    .reindex(dfUsuario["userid"]).fillna(0).values
)
dfUsuario["casos_comparacion_arriba"] = (
    dfNivelEjercicio[mask_arriba].groupby("userid").size()
    .reindex(dfUsuario["userid"]).fillna(0).values
)
dfUsuario["casos_comparacion_abajo"] = (
    dfNivelEjercicio[mask_abajo].groupby("userid").size()
    .reindex(dfUsuario["userid"]).fillna(0).values
)

# Performance global
dfUsuario["performance"] = 0.0
mask = den > 0
dfUsuario.loc[mask, "performance"] = num[mask] / den[mask]

# Performance arriba
agg_arriba = (
    dfNivelEjercicio[mask_arriba].groupby("userid").agg(
        fac_sum=("first_attempts_correct", "sum"),
        us_sum=("unique_steps", "sum"),
    )
)
agg_arriba["performance_arriba"] = (
    agg_arriba["fac_sum"] / agg_arriba["us_sum"].replace(0, np.nan)
)
dfUsuario["performance_arriba"] = (
    dfUsuario["userid"].map(agg_arriba["performance_arriba"]).fillna(0)
)

# Performance abajo
agg_abajo = (
    dfNivelEjercicio[mask_abajo].groupby("userid").agg(
        fac_sum=("first_attempts_correct", "sum"),
        us_sum=("unique_steps", "sum"),
    )
)
agg_abajo["performance_abajo"] = (
    agg_abajo["fac_sum"] / agg_abajo["us_sum"].replace(0, np.nan)
)
dfUsuario["performance_abajo"] = (
    dfUsuario["userid"].map(agg_abajo["performance_abajo"]).fillna(0)
)

# Tiempo arriba / abajo
tiempo_arriba = dfNivelEjercicio[mask_arriba].groupby("userid")["duracion_segundos"].sum()
tiempo_abajo = dfNivelEjercicio[mask_abajo].groupby("userid")["duracion_segundos"].sum()
dfUsuario["tiempo_casos_comparacion_arriba"] = dfUsuario["userid"].map(tiempo_arriba).fillna(0)
dfUsuario["tiempo_casos_comparacion_abajo"] = dfUsuario["userid"].map(tiempo_abajo).fillna(0)

# Porcentajes
dfUsuario["porce_topicos_arriba"] = dfUsuario["casos_comparacion_arriba"] / dfUsuario["casos"]
dfUsuario["porce_topicos_abajo"] = dfUsuario["casos_comparacion_abajo"] / dfUsuario["casos"]

print("Comparaciones y performance calculados.")

## 4.2 Métricas de autoeficacia/progreso/rating por dirección

In [ ]:
dfNivelEjercicio['pollValue_num'] = pd.to_numeric(
    dfNivelEjercicio['pollValue_num'], errors='coerce'
)
dfNivelEjercicio['selectionRating_result'] = pd.to_numeric(
    dfNivelEjercicio['selectionRating_result'], errors='coerce'
)

df_upward = dfNivelEjercicio[
    (dfNivelEjercicio['last_progressGroup'] != -1) &
    ((dfNivelEjercicio['last_progressMe'] - dfNivelEjercicio['last_progressGroup']) >= 0.1)
]
df_downward = dfNivelEjercicio[
    (dfNivelEjercicio['last_progressGroup'] != -1) &
    ((dfNivelEjercicio['last_progressMe'] - dfNivelEjercicio['last_progressGroup']) <= -0.1)
]

def calcular_metricas(df):
    return df.groupby('userid').agg(
        tasa_exito=('first_attempts_correct', lambda x: x.sum() / len(x) if len(x) > 0 else np.nan),
        autoeficacia_prom=('pollValue_num', 'mean'),
        progreso_inicial_prom=('last_progressMe', 'mean'),
        rating=('selectionRating_result', 'mean')
    )

metricas_upward = calcular_metricas(df_upward).add_suffix('_comparacion_arriba')
metricas_downward = calcular_metricas(df_downward).add_suffix('_comparacion_abajo')

dfUsuario = (
    dfUsuario.set_index('userid')
    .join(metricas_upward)
    .join(metricas_downward)
    .reset_index()
)

# Filtros de outliers
dfUsuario = dfUsuario[
    (dfUsuario['total_attempts'] < 600) &
    (dfUsuario['numero_sesiones_diferentes'] < 15)
]

print(f"dfUsuario final: {len(dfUsuario)} usuarios (post filtro outliers)")

## 4.3 Proporciones y preferencias adicionales

In [ ]:
dfUsuario['proporcion_casos_comparacion'] = (
    (dfUsuario['casos_comparacion_arriba'] + dfUsuario['casos_comparacion_abajo'])
    / dfUsuario['casos']
)
dfUsuario['proporcion_steps_comparacion'] = (
    (dfUsuario['pasos_comparacion_arriba'] + dfUsuario['pasos_comparacion_abajo'])
    / dfUsuario['total_attempts']
)
dfUsuario['proporcion_casos_uw'] = dfUsuario['casos_comparacion_arriba'] / dfUsuario['casos']
dfUsuario['proporcion_steps_uw'] = dfUsuario['pasos_comparacion_arriba'] / dfUsuario['total_attempts']
dfUsuario['proporcion_casos_dw'] = dfUsuario['casos_comparacion_abajo'] / dfUsuario['casos']
dfUsuario['proporcion_pasos_dw'] = dfUsuario['pasos_comparacion_abajo'] / dfUsuario['total_attempts']

dfUsuario['preferencia_casos_comparacion'] = (
    (dfUsuario['casos_comparacion_arriba'] + dfUsuario['casos_comparacion_abajo'])
    - (dfUsuario['casos'] - dfUsuario['casos_comparacion_arriba'] - dfUsuario['casos_comparacion_abajo'])
) / dfUsuario['casos']

dfUsuario['preferencia_casos_uw'] = (
    (dfUsuario['casos_comparacion_arriba'] - dfUsuario['casos_comparacion_abajo'])
    / (dfUsuario['casos_comparacion_arriba'] + dfUsuario['casos_comparacion_abajo'])
)
dfUsuario['preferencia_steps_uw'] = (
    (dfUsuario['pasos_comparacion_arriba'] - dfUsuario['pasos_comparacion_abajo'])
    / (dfUsuario['pasos_comparacion_arriba'] + dfUsuario['pasos_comparacion_abajo'])
)
dfUsuario['preferencia_pasos_comparacion'] = (
    (dfUsuario['pasos_comparacion_arriba'] + dfUsuario['pasos_comparacion_abajo'])
    - (dfUsuario['total_attempts'] - dfUsuario['pasos_comparacion_arriba'] - dfUsuario['pasos_comparacion_abajo'])
) / dfUsuario['total_attempts']

dfUsuario['tupdivtdownpasos'] = (
    (dfUsuario['pasos_comparacion_arriba'] + 1) / (dfUsuario['pasos_comparacion_abajo'] + 1)
)

print(f"dfUsuario completo: {len(dfUsuario)} usuarios, {len(dfUsuario.columns)} columnas")
dfUsuario.groupby('grupo_intervencion')['performance'].describe()

# 5. Construcción de dfSegmento

**Entrada**: `dfNivelEjercicio` (completados)  
**Proceso**: Consolidar métricas por (userid, topiclabel, pollCount), crear transiciones entre mediciones de autoeficacia, rellenar pollCounts faltantes y generar pares de transición.

In [ ]:
df = dfNivelEjercicio.sort_values(
    ['userid', 'topiclabel', 'pollCount', 'session']
).copy()

df[['last_progressGroup', 'last_progressMe']] = (
    df.groupby(['userid', 'topiclabel'])[['last_progressGroup', 'last_progressMe']].ffill()
)

for col in ['last_progressGroup', 'last_progressMe']:
    df[col] = pd.to_numeric(df[col], errors='coerce').clip(0, 1).astype('float64')

to_num = lambda s: pd.to_numeric(s, errors='coerce')
df['pollCount'] = to_num(df['pollCount']).astype('Int64')
df['pollValue'] = to_num(df['pollValue'])
df['total_attempts'] = to_num(df['total_attempts'])
df['first_attempts_correct'] = to_num(df['first_attempts_correct'])
df['unique_steps'] = to_num(df['unique_steps'])
df['selectionRating_result'] = to_num(df['selectionRating_result']).astype('float64')

df['prop_first_attempts_correct'] = np.where(
    df['unique_steps'] > 0,
    df['first_attempts_correct'] / df['unique_steps'],
    np.nan
)

df = df[df['pollCount'].ge(1)]

topic_max = df.groupby(['userid', 'topiclabel'])['pollCount'].max().reset_index()
topic_validos = topic_max[topic_max['pollCount'].ge(2)][['userid', 'topiclabel']]
df = df.merge(topic_validos, on=['userid', 'topiclabel'], how='inner')

last_notna = lambda s: s.dropna().iloc[-1] if s.notna().any() else pd.NA

df_sess = (
    df.groupby(['userid', 'topiclabel', 'pollCount', 'session'], as_index=False)
    .agg({
        'pollValue': 'mean',
        'total_attempts': 'sum',
        'first_attempts_correct': 'sum',
        'unique_steps': 'sum',
        'prop_first_attempts_correct': 'mean',
        'last_progressGroup': last_notna,
        'last_progressMe': last_notna,
        'selectionRating_result': last_notna,
        'grupo_intervencion': 'first',
        'direccionComparacion': 'first',
    })
)

for col in ['last_progressGroup', 'last_progressMe']:
    df_sess[col] = pd.to_numeric(df_sess[col], errors='coerce').clip(0, 1).astype('float64')
df_sess['selectionRating_result'] = pd.to_numeric(df_sess['selectionRating_result'], errors='coerce').astype('float64')
df_sess['prop_first_attempts_correct'] = pd.to_numeric(df_sess['prop_first_attempts_correct'], errors='coerce').astype('float64')

df_unique = (
    df_sess.sort_values(['userid', 'topiclabel', 'pollCount', 'session'])
    .groupby(['userid', 'grupo_intervencion', 'topiclabel', 'pollCount'], as_index=False)
    .agg({
        'pollValue': 'mean',
        'total_attempts': 'sum',
        'first_attempts_correct': 'sum',
        'unique_steps': 'sum',
        'prop_first_attempts_correct': 'mean',
        'last_progressGroup': 'last',
        'last_progressMe': 'last',
        'selectionRating_result': 'first',
        'direccionComparacion': 'first',
        'session': 'min',
    })
)

for col in ['last_progressGroup', 'last_progressMe']:
    df_unique[col] = pd.to_numeric(df_unique[col], errors='coerce').clip(0, 1).astype('float64')
df_unique['selectionRating_result'] = pd.to_numeric(df_unique['selectionRating_result'], errors='coerce').astype('float64')
df_unique['prop_first_attempts_correct'] = pd.to_numeric(df_unique['prop_first_attempts_correct'], errors='coerce').astype('float64')

# FIX: Iterar manualmente para evitar pérdida de columnas de agrupación en pandas >=2.x
all_seq = []
for (user_id, topic, interv, direccion), grupo in df_unique.groupby(
    ['userid', 'topiclabel', 'grupo_intervencion', 'direccionComparacion'], sort=False
):
    max_pc = int(pd.to_numeric(grupo['pollCount'], errors='coerce').max())
    todos = pd.DataFrame({'pollCount': range(1, max_pc + 1)})
    columnas_no_clave = [
        c for c in grupo.columns
        if c not in ['userid', 'topiclabel', 'grupo_intervencion', 'direccionComparacion', 'pollCount']
    ]
    out = todos.merge(grupo[['pollCount'] + columnas_no_clave], on='pollCount', how='left')
    out['userid'] = user_id
    out['topiclabel'] = topic
    out['grupo_intervencion'] = interv
    out['direccionComparacion'] = direccion
    out['pollCount'] = out['pollCount'].astype('Int64')
    all_seq.append(out)
df_seq = pd.concat(all_seq, ignore_index=True)

for col in ['last_progressGroup', 'last_progressMe']:
    df_seq[col] = pd.to_numeric(df_seq[col], errors='coerce').clip(0, 1).astype('float64')
df_seq['selectionRating_result'] = pd.to_numeric(df_seq['selectionRating_result'], errors='coerce').astype('float64')
df_seq['prop_first_attempts_correct'] = pd.to_numeric(df_seq['prop_first_attempts_correct'], errors='coerce').astype('float64')

df_seq = df_seq.sort_values(['userid', 'topiclabel', 'pollCount'])
df_seq['pollCount_next'] = df_seq.groupby(['userid', 'topiclabel'])['pollCount'].shift(-1).astype('Int64')
df_seq['pollValue_next'] = df_seq.groupby(['userid', 'topiclabel'])['pollValue'].shift(-1)

dfSegmento = df_seq[[
    'userid', 'topiclabel', 'pollCount', 'session', 'pollValue',
    'total_attempts', 'first_attempts_correct', 'unique_steps',
    'prop_first_attempts_correct',
    'last_progressGroup', 'last_progressMe',
    'selectionRating_result', 'grupo_intervencion', 'direccionComparacion',
    'pollCount_next', 'pollValue_next'
]].reset_index(drop=True)

dfSegmento['last_progressGroup'] = dfSegmento['last_progressGroup'].replace(-1, np.nan)

print(f"dfSegmento creado: {len(dfSegmento)} filas")

## 5.1 Recalcular dirección de comparación en dfSegmento

Dirección corregida:  
- `progressMe > progressGroup` → estudiante está ARRIBA del grupo → **upward**  
- `progressMe < progressGroup` → estudiante está ABAJO del grupo → **downward**

In [ ]:
dfSegmento['grupo_intervencion'] = dfSegmento['grupo_intervencion'].astype(str).str.strip()
cols_prog = ['last_progressMe', 'last_progressGroup']
dfSegmento[cols_prog] = dfSegmento[cols_prog].apply(pd.to_numeric, errors='coerce')

no_cmp = (
    dfSegmento['grupo_intervencion'].eq('OLM') |
    dfSegmento['last_progressMe'].eq(-1) |
    dfSegmento['last_progressGroup'].eq(-1) |
    dfSegmento[cols_prog].isna().any(axis=1)
)

diff = dfSegmento['last_progressMe'] - dfSegmento['last_progressGroup']

dfSegmento['direccionComparacion'] = np.select(
    [no_cmp, diff.ge(0.1), diff.le(-0.1), diff.abs().lt(0.1)],
    ['no', 'upward', 'downward', 'equal'],
    default='no'
)

dfSegmento = dfSegmento[
    ~(
        dfSegmento['grupo_intervencion'].isin(['OSLM', 'OSLM+VP']) &
        (dfSegmento['direccionComparacion'] == 'no')
    )
]

dfSegmento['delta_pollValue'] = dfSegmento['pollValue_next'] - dfSegmento['pollValue']

print(f"dfSegmento final: {len(dfSegmento)} filas")
print(dfSegmento['direccionComparacion'].value_counts())

## 5.2 Transiciones entre mediciones

In [ ]:
def transition_dfs(df, poll_col='pollCount', pollnext_col='pollCount_next',
                   min_from=1, key_cols=None, pairs=None):
    if key_cols is None:
        key_cols = ['userid', 'topiclabel']
    df2 = df[df[pollnext_col].notna()].copy()
    df2[poll_col] = pd.to_numeric(df2[poll_col], errors='coerce').astype('Int64')
    df2[pollnext_col] = pd.to_numeric(df2[pollnext_col], errors='coerce').astype('Int64')
    if pairs is None:
        max_n = int(max(df2[poll_col].max(), df2[pollnext_col].max()))
        pairs = [(n, n + 1) for n in range(min_from, max_n)]
    transitions = {}
    for a, b in pairs:
        mask = (df2[poll_col] == a) & (df2[pollnext_col] == b)
        dfn = df2.loc[mask].copy()
        if not dfn.empty:
            dfn = dfn[dfn[key_cols].notna().all(axis=1)]
        transitions[f'{a}->{b}'] = dfn.reset_index(drop=True)
    return transitions

pairs = [(1, 2), (2, 3), (1, 3)]
transitions = transition_dfs(dfSegmento, pairs=pairs)
df_1_to_2 = transitions['1->2']
df_2_to_3 = transitions['2->3']
df_1_to_3 = transitions['1->3']

print(f"Transiciones: 1->2={len(df_1_to_2)}, 2->3={len(df_2_to_3)}, 1->3={len(df_1_to_3)}")

# 6. Exportación de archivos

In [ ]:
output_folder = os.path.dirname(os.path.abspath("__file__"))
os.makedirs(output_folder, exist_ok=True)

df_ej_export = dfNivelEjercicio.drop(columns=["trayectoria", "poll_history_topic"], errors="ignore")

df_ej_export.to_excel(os.path.join(output_folder, "dfNivelEjercicio.xlsx"), index=False)
dfUsuario.to_excel(os.path.join(output_folder, "dfUsuario.xlsx"), index=False)
dfSegmento.to_excel(os.path.join(output_folder, "dfSegmento.xlsx"), index=False)
df_1_to_2.to_excel(os.path.join(output_folder, "df_1_to_2.xlsx"), index=False)
df_2_to_3.to_excel(os.path.join(output_folder, "df_2_to_3.xlsx"), index=False)

print(f"Archivos guardados en: {output_folder}")
print("  - dfNivelEjercicio.xlsx")
print("  - dfUsuario.xlsx")
print("  - dfSegmento.xlsx")
print("  - df_1_to_2.xlsx")
print("  - df_2_to_3.xlsx")
print("\nPreprocesamiento completado!")

In [ ]:
dfNivelEjercicio[dfNivelEjercicio['pollValue'].notna()]['userid'].unique()

In [ ]:
dfNivelEjercicio[dfNivelEjercicio['userid'] == 2000]